Gulzhan Aitileu

Key Insights & Take-aways:
- Data Preprocessing is 90% of the Work. Handling a space-separated dataset with no headers required careful initial loading. Converting 20 raw features into 61 numerical features via One-Hot Encoding was essential for the Neural Network to process categorical data like "Housing" and "Job" status.

- Preventing Data Leakage: A major takeaway was the importance of splitting the data before scaling. By fitting the StandardScaler only on the training set, I ensured that the model remained truly blind to the test data, providing a realistic evaluation of its performance.

- The Precision-Recall Trade-off: While the model achieved a solid 76% overall accuracy, the classification report revealed that it is much better at identifying "Good" customers (84% recall) than "Bad" ones (56% recall).

- Business Impact. In a real-world banking scenario, the "cost" of a missed default (False Positive) is often higher than the cost of a missed customer (False Negative). This project highlighted that accuracy alone is a misleading metric for imbalanced datasets; monitoring the F1-Score and Recall for the "Bad Credit" class is critical for managing financial risk.

- PyTorch Workflow. Completing the end-to-end cycle (from TensorDatasets and DataLoaders to the Forward Pass and Backpropagation) solidified my understanding of how gradients are used to optimize weights in a multi-layer perceptron.

In [42]:
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
# Accessing the data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
# Defining column names
column_names = [
    "Status of existing checking account",
    "Duration in month",
    "Credit history",
    "Purpose",
    "Credit amount",
    "Savings account/bonds",
    "Present employment since",
    "Installment rate in percentage of disposable income",
    "Personal status and sex",
    "Other debtors / guarantors",
    "Present residence since",
    "Property",
    "Age in years",
    "Other installment plans",
    "Housing",
    "Number of existing credits at this bank",
    "Job",
    "Number of people being liable to provide maintenance for",
    "Telephone",
    "foreign worker",
    "Creditability"
]

In [45]:
# Accessing the FILE from Google Drive
df = pd.read_csv('/content/drive/MyDrive/S0304405X25001011.csv', sep=r'\s+', names=column_names)

In [46]:
# Target Column
df['Creditability'] = df['Creditability'].replace({1: 1, 2: 0})

In [47]:
print("Data Loaded Successfully!")
print(df.head(3))

Data Loaded Successfully!
  Status of existing checking account  Duration in month Credit history  \
0                                 A11                  6            A34   
1                                 A12                 48            A32   
2                                 A14                 12            A34   

  Purpose  Credit amount Savings account/bonds Present employment since  \
0     A43           1169                   A65                      A75   
1     A43           5951                   A61                      A73   
2     A46           2096                   A61                      A74   

   Installment rate in percentage of disposable income  \
0                                                  4     
1                                                  2     
2                                                  2     

  Personal status and sex Other debtors / guarantors  ...  Property  \
0                     A93                       A101  ...      A121 

Preprocessing

In [48]:
# Preprocessing data
# 1. Identify your target and features
# We want to encode the features, not the target label
X = df.drop('Creditability', axis=1)
y = df['Creditability']


In [49]:
# 2. Apply One-Hot Encoding
# columns=None tells pandas to find all categorical columns automatically
X_encoded = pd.get_dummies(X)

In [50]:
print(f"Original features count: {X.shape[1]}")
print(f"Features after One-Hot Encoding: {X_encoded.shape[1]}")
print(X_encoded.head())

Original features count: 20
Features after One-Hot Encoding: 61
   Duration in month  Credit amount  \
0                  6           1169   
1                 48           5951   
2                 12           2096   
3                 42           7882   
4                 24           4870   

   Installment rate in percentage of disposable income  \
0                                                  4     
1                                                  2     
2                                                  2     
3                                                  2     
4                                                  3     

   Present residence since  Age in years  \
0                        4            67   
1                        2            22   
2                        3            49   
3                        4            45   
4                        4            53   

   Number of existing credits at this bank  \
0                                        

Standardizing

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [52]:
# 1. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42
)

In [53]:
# 2. Initialize the Scaler
scaler = StandardScaler()

In [54]:
# 3. Fit the scaler ONLY on the training data
# This learns the mean and standard deviation of the training set
scaler.fit(X_train)

StandardScaler()

In [55]:
# 4. Transform both the training and testing data
# We use the training set's "rules" to scale the test set
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [56]:
print(f"Training set size: {X_train_scaled.shape[0]} rows")
print(f"Testing set size: {X_test_scaled.shape[0]} rows")

Training set size: 800 rows
Testing set size: 200 rows


Create DataLoaders

In [57]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [58]:
# 1. Convert NumPy arrays/Pandas Series to PyTorch Tensors
# We use float32 for features and float32 for labels (standard for BCELoss)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [59]:
# 2. Combine features and labels into a Dataset object
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [60]:
# 3. Create DataLoaders
# We shuffle the training data so the model doesn't learn the order of the rows
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"DataLoaders created!")
print(f"Number of training batches: {len(train_loader)}")

DataLoaders created!
Number of training batches: 25


Build an MLP

In [61]:
import torch.nn as nn

class LoanDefaultMLP(nn.Module):
    def __init__(self, input_size):
        super(LoanDefaultMLP, self).__init__()

        # Layer 1: Input -> Hidden 1
        # input_size is the number of features in X_train
        self.hidden1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()

        # Layer 2: Hidden 1 -> Hidden 2
        self.hidden2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()

        # Layer 3: Output Layer (Hidden 2 -> 1 Output)
        self.output = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # This defines how data flows through the layers
        x = self.hidden1(x)
        x = self.relu1(x)
        x = self.hidden2(x)
        x = self.relu2(x)
        x = self.output(x)
        x = self.sigmoid(x)
        return x

# Initialize the model
# input_size is the number of columns in our scaled data
input_dim = X_train_scaled.shape[1]
model = LoanDefaultMLP(input_dim)

print(model)

LoanDefaultMLP(
  (hidden1): Linear(in_features=61, out_features=64, bias=True)
  (relu1): ReLU()
  (hidden2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (output): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


Training the model

In [62]:
import torch.optim as optim

# 1. Define Loss Function and Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. The Training Loop
epochs = 50

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        # Zero the gradients so they don't accumulate
        optimizer.zero_grad()

        # Forward pass: Get predictions
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass: Calculate gradients
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")

Epoch [10/50], Loss: 0.3509
Epoch [20/50], Loss: 0.1325
Epoch [30/50], Loss: 0.0389
Epoch [40/50], Loss: 0.0128
Epoch [50/50], Loss: 0.0068
Training Complete!


Evaluation

In [63]:
# 1. Set model to evaluation mode
model.eval()

correct = 0
total = 0

In [64]:
# 2. No need to track gradients during evaluation
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)

        # Convert probabilities (0-1) to binary predictions (0 or 1)
        # If probability > 0.5, we predict 1 (Good), else 0 (Bad)
        predictions = (outputs > 0.5).float()

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

In [65]:
# 3. Calculate and print accuracy
accuracy = 100 * correct / total
print(f"Final Accuracy on Test Set: {accuracy:.2f}%")

Final Accuracy on Test Set: 73.50%


Reporting: Precision, Recall, F1-Score

In [66]:
from sklearn.metrics import classification_report

# 1. Collect all predictions and true labels
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predictions = (outputs > 0.5).float()

        all_preds.extend(predictions.numpy())
        all_labels.extend(labels.numpy())

# 2. Print the detailed report
# Target names: 0 is 'Bad Credit', 1 is 'Good Credit'
print("Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=['Bad Credit (0)', 'Good Credit (1)']))

Classification Report:

                 precision    recall  f1-score   support

 Bad Credit (0)       0.55      0.58      0.56        59
Good Credit (1)       0.82      0.80      0.81       141

       accuracy                           0.73       200
      macro avg       0.68      0.69      0.69       200
   weighted avg       0.74      0.73      0.74       200



In [67]:
from sklearn.metrics import precision_recall_fscore_support

# 1. Get the metrics numerically so we can interpret them
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None)

# 2. Print the Interpretation
print("-" * 30)
print("🏦 MODEL INTERPRETATION 🏦")
print("-" * 30)

# Interpreting "Bad Credit" (Class 0)
print(f"1. RISK DETECTION (Recall for Bad Credit): {recall[0]*100:.1f}%")
print(f"   Interpretation: Your model catches {recall[0]*100:.1f}% of people who will actually default.")
print(f"   The 'Miss Rate' is {(1-recall[0])*100:.1f}%—these are risky loans the bank would accidentally approve.")

print(f"\n2. LOAN APPROVAL ACCURACY (Precision for Good Credit): {precision[1]*100:.1f}%")
print(f"   Interpretation: When the model says 'Yes' to a loan, it is correct {precision[1]*100:.1f}% of the time.")

# Overall Summary
print("\n3. OVERALL SUMMARY:")
if recall[0] < 0.5:
    print("   ⚠️ WARNING: The model is struggling to identify 'Bad' customers. It's playing it too safe.")
elif recall[0] > recall[1]:
    print("   🛡️ STRENGTH: The model is very conservative and good at spotting risky loans.")
else:
    print("   ⚖️ BALANCE: The model has a decent balance between approving loans and catching defaults.")
print("-" * 30)

------------------------------
🏦 MODEL INTERPRETATION 🏦
------------------------------
1. RISK DETECTION (Recall for Bad Credit): 57.6%
   Interpretation: Your model catches 57.6% of people who will actually default.
   The 'Miss Rate' is 42.4%—these are risky loans the bank would accidentally approve.

2. LOAN APPROVAL ACCURACY (Precision for Good Credit): 81.9%
   Interpretation: When the model says 'Yes' to a loan, it is correct 81.9% of the time.

3. OVERALL SUMMARY:
   ⚖️ BALANCE: The model has a decent balance between approving loans and catching defaults.
------------------------------
